# CV refinement of the 0PA + simple-deviation MAP

For each case we start from the **CV-refined 0PA MAP with `dev_1 = dev_2 = 0`** (11 params:
`m1, m2, a, p0, e0, qS, phiS, Phi_phi0, Phi_r0, dev_1, dev_2`). These 0PA points already sit at
overlap > 0.999 for most cases, so the initial overlap = the 0PA overlap; we then let the two
simple-deviation DOF (`dev_1, dev_2`) turn on and test whether adaptive Levenberg-Marquardt CV
steps (Fisher gradient) can **improve** the overlap with the 1PA signal.

Simple deviation enters `SuperKludgeFlux` as the multiplicative flux rescaling
`Edot -> (1 + eta*del_0_p) Edot`, `Ldot -> (1 + eta*del_0_e) Ldot`, i.e.
`additional_args = [chi2, evolve_1PA, evolve_primary, evolve_2PA, deviation_included,
C_p, C_e, del_0_p, del_0_e]` with **`dev_1 = del_0_p` (idx 7)**, **`dev_2 = del_0_e` (idx 8)**
and `deviation_included = True`. No Nelder-Mead here -- pure CV.


In [1]:
import numpy as np

from few.waveform import GenerateEMRIWaveform
from few.waveform.waveform import SuperKludgeWaveform
from fastlisaresponse import ResponseWrapper
from lisatools.detector import EqualArmlengthOrbits
from lisatools.sensitivity import get_sensitivity, A1TDISens, E1TDISens, T1TDISens
from stableemrifisher.utils import generate_PSD, inner_product, fishinv
from stableemrifisher.fisher import StableEMRIFisher

try:
    import cupy as cp
    xp = cp
except ImportError:
    xp = np
    print("[INFO] CuPy not found, using NumPy instead.")

F_MIN = 1e-5


def _to_float(x):
    return float(x.get()) if hasattr(x, "get") else float(x)


def make_freq_mask(n, dt, fmin):
    return (xp.fft.rfftfreq(n, dt) > fmin)[1:]


def highpass_clip(w, dt, fmin):
    n = w.shape[-1]
    f = xp.fft.rfftfreq(n, dt)
    return xp.fft.irfft(xp.fft.rfft(w, axis=-1) * (f >= fmin), n=n, axis=-1)


startup


In [2]:
# --- CV / adaptive Levenberg-Marquardt controls ---
NDELTA = 24                    # SEF stability-search grid size
RECOMPUTE_DELTAS_EVERY = 5     # re-optimise finite-diff steps every N points
OVERLAP_TARGET = 0.999999        # stop the climb once overlap exceeds this
LAMBDA0, MAX_ITERS, MAX_INNER, REL_TOL = 1e-2, 120, 30, 1e-8

use_gpu = True
nchannels = 3
param_names_14 = ["m1", "m2", "a", "p0", "e0", "xI0", "dist", "qS", "phiS",
                  "qK", "phiK", "Phi_phi0", "Phi_theta0", "Phi_r0"]
# 11 inferred: the 9 EMRI params + the two simple-deviation coefficients.
params_to_infer = ["m1", "m2", "a", "p0", "e0", "qS", "phiS", "Phi_phi0", "Phi_r0",
                   "dev_1", "dev_2"]

# Each case: the 1PA injected truth (dev=0) plus the NM 0PA+simple-dev start point (11-vec).
CASES = [
    dict(name="idx0", dt=5.0, T=1.0, chi2=0.0, dev_nm_overlap=0.9982902463962956,
         signal_param={"m1": 1e6, "m2": 10.0, "a": 0.9, "p0": 7.5, "e0": 0.5,
                       "xI0": 1.0, "dist": 5.0, "qS": 0.7853981633974483, "phiS": 1.0,
                       "qK": 1.0, "phiK": 1.0471975511965976, "Phi_phi0": 0.9,
                       "Phi_theta0": 0.5, "Phi_r0": 0.4, "dev_1": 0.0, "dev_2": 0.0},
         dev_start=[1.00014867e+06, 1.00010513e+01, 9.00096370e-01, 7.49942614e+00,
                    4.99974688e-01, 7.83388290e-01, 9.99626083e-01, 9.09269038e-01,
                    3.96038198e-01, 0.0, 0.0]),  # CV-refined 0PA MAP + dev=0
    dict(name="idx9", dt=10.0, T=2.5, chi2=0.95, dev_nm_overlap=0.9980454212618244,
         signal_param={"m1": 1e6, "m2": 10.0, "a": 0.9, "p0": 9.07414088, "e0": 0.2,
                       "xI0": 1.0, "dist": 5.0, "qS": 1.04719755, "phiS": 0.785398163,
                       "qK": 0.628318531, "phiK": 0.523598776, "Phi_phi0": 0.1,
                       "Phi_theta0": 0.2, "Phi_r0": 0.3, "dev_1": 0.0, "dev_2": 0.0},
         dev_start=[9.99994929e+05, 1.00000755e+01, 9.00001697e-01, 9.07417360e+00,
                    1.99997410e-01, 1.04653506e+00, 7.83187637e-01, 1.59230827e-01,
                    9.90108700e-02, 0.0, 0.0]),  # CV-refined 0PA MAP + dev=0
    dict(name="idx13", dt=10.0, T=2.5, chi2=0.95, dev_nm_overlap=0.9998873034358231,
         signal_param={"m1": 1e6, "m2": 10.0, "a": 0.5, "p0": 9.97066819, "e0": 0.3,
                       "xI0": 1.0, "dist": 5.0, "qS": 1.04719755, "phiS": 0.785398163,
                       "qK": 0.628318531, "phiK": 0.523598776, "Phi_phi0": 0.1,
                       "Phi_theta0": 0.2, "Phi_r0": 0.3, "dev_1": 0.0, "dev_2": 0.0},
         dev_start=[1.00003623e+06, 1.00001339e+01, 5.00052129e-01, 9.97041651e+00,
                    2.99991839e-01, 1.04533235e+00, 7.83864901e-01, 1.13205146e-01,
                    2.54180379e-01, 0.0, 0.0]),  # CV-refined 0PA MAP + dev=0
]


In [3]:
def run_case(case, verbose=True):
    sp, dt, T, chi2 = case["signal_param"], case["dt"], case["T"], case["chi2"]
    channels = [A1TDISens, E1TDISens, T1TDISens][:nchannels]
    tdi_chan = {2: "AE", 3: "AET"}[nchannels]
    noise_kwargs = [{"sens_fn": ch} for ch in channels]

    def rkw():
        return dict(Tobs=T, t0=10000.0, dt=dt, index_lambda=8, index_beta=7, flip_hx=True,
                    is_ecliptic_latitude=False, remove_garbage="zero",
                    orbits=EqualArmlengthOrbits(use_gpu=use_gpu),
                    force_backend="cuda12x" if use_gpu else "cpu",
                    order=20, tdi="1st generation", tdi_chan=tdi_chan)

    wfm = GenerateEMRIWaveform(SuperKludgeWaveform,
                               sum_kwargs=dict(pad_output=True, odd_len=True),
                               return_list=False, use_gpu=use_gpu)
    wresp = ResponseWrapper(waveform_gen=wfm, **rkw())

    def resp_args(vec, evolve_1pa, deviation_on):
        """14 EMRI params + [chi2, evolve_1PA, evolve_primary, evolve_2PA,
        deviation_included, C_p, C_e, del_0_p(=dev_1), del_0_e(=dev_2)]."""
        p = {n: sp[n] for n in param_names_14}
        p.update(dict(zip(params_to_infer, vec)))          # overwrite the 9 EMRI params
        return [p[n] for n in param_names_14] + [
            chi2, evolve_1pa, False, False, deviation_on, 0.0, 0.0, vec[9], vec[10]]

    def make0(vec):                                        # 0PA + simple-deviation template
        return xp.array(wresp(*resp_args(vec, False, True)))[:nchannels, :]

    # 1PA injected signal (deviation off, dev = 0)
    true_inf = np.array([sp[n] for n in params_to_infer])
    s = highpass_clip(xp.array(wresp(*resp_args(true_inf, True, False)))[:nchannels, :], dt, F_MIN)
    PSD = xp.array(generate_PSD(waveform=s, dt=dt, noise_PSD=get_sensitivity,
                                channels=channels, noise_kwargs=noise_kwargs, use_gpu=use_gpu))
    fmask = make_freq_mask(s.shape[-1], dt, F_MIN)

    def ip(a, b):
        return _to_float(inner_product(a, b, PSD=PSD, dt=dt, freq_mask=fmask, use_gpu=use_gpu))

    def ov(vec):
        h = make0(vec)
        return ip(s, h) / np.sqrt(ip(s, s) * ip(h, h))

    def chi2r(vec):
        try:
            r = s - make0(vec)
            return ip(r, r)
        except Exception:
            return 1e30

    sef = StableEMRIFisher(
        waveform_class=SuperKludgeWaveform,
        waveform_class_kwargs=dict(sum_kwargs=dict(pad_output=True, odd_len=True)),
        waveform_generator=GenerateEMRIWaveform,
        waveform_generator_kwargs=dict(return_list=False),
        ResponseWrapper=ResponseWrapper, ResponseWrapper_kwargs=rkw(),
        stats_for_nerds=False, use_gpu=use_gpu, deriv_type="stable",
        noise_model=get_sensitivity, noise_kwargs=noise_kwargs, channels=channels,
        T=T, dt=dt, stability_plot=False, der_order=6, Ndelta=NDELTA,
        plunge_check=True, return_derivatives=True)

    def fisher_derivs(vec, dl):
        """0PA+simple-dev Fisher and derivatives at the 11-vector `vec`."""
        wp = {n: sp[n] for n in param_names_14}
        wp.update(dict(zip(params_to_infer, vec)))
        apa = {"chi2": chi2, "evolve_1PA": False, "evolve_primary": False,
               "evolve_2PA": False, "deviation_included": True,
               "C_p": 0.0, "C_e": 0.0, "dev_1": vec[9], "dev_2": vec[10]}
        F = sef(wave_params={n: wp[n] for n in param_names_14}, param_names=params_to_infer,
                add_param_args=apa, deltas=dl, live_dangerously=False, stability_plot=False,
                der_order=8, Ndelta=(NDELTA if dl is None else None))
        return np.asarray(F[-1], dtype=float), xp.array(F[0]), sef.deltas

    # ---- adaptive Levenberg-Marquardt CV climb from the NM simple-dev start ----
    npar = len(params_to_infer)
    cur = np.array(case["dev_start"], dtype=float)
    ov_start = ov(cur)
    snr = np.sqrt(ip(s, s))
    if verbose:
        print(f"[{case['name']}] SNR={snr:.2f}  overlap(start, NM ov={case['dev_nm_overlap']:.6f}) = {ov_start:.6f}")

    lam, nu, dl, converged = LAMBDA0, 2.0, None, False
    for it in range(MAX_ITERS):
        if it % RECOMPUTE_DELTAS_EVERY == 0:
            dl = None
        G, dH, deltas = fisher_derivs(cur, dl)
        if dl is None:
            dl = deltas
        h = make0(cur)
        r = s - h
        g = np.array([ip(dH[j], r) for j in range(npar)])
        sig = np.sqrt(np.abs(np.diag(fishinv(cur[0], G, index_of_M=0))))
        c0, ovc = ip(r, r), ip(s, h) / np.sqrt(ip(s, s) * ip(h, h))
        if ovc > OVERLAP_TARGET:
            converged = True
            break

        dvec = np.abs(np.diag(G)) + 1e-30
        delta, ok, rel = np.zeros(npar), False, 0.0
        for _ in range(MAX_INNER):
            try:
                delta = np.linalg.solve(G + lam * np.diag(dvec), g)
            except np.linalg.LinAlgError:
                lam *= nu; nu *= 2.0; continue
            pred = float(delta @ (g + lam * dvec * delta))
            rho = (c0 - chi2r(cur + delta)) / pred if pred > 0 else -1.0
            if rho > 0.0:
                lam *= max(1.0 / 3.0, 1.0 - (2.0 * rho - 1.0) ** 3); nu = 2.0
                rel = (c0 - chi2r(cur + delta)) / c0; ok = True; break
            lam *= nu; nu *= 2.0
        if verbose:
            print(f"[{case['name']}]  it {it:>3} lam={lam:.1e} ov={ovc:.7f} chi2={c0:.3e} "
                  f"|d/sig|={float(np.max(np.abs(delta / sig))):.2e} rel={rel:.1e}")
        if not ok:
            break
        cur = cur + delta
        if rel < REL_TOL:
            converged = True
            break

    return dict(name=case["name"], snr=float(snr), ov_start=float(ov_start),
                ov_final=float(ov(cur)), dev_nm_overlap=case["dev_nm_overlap"],
                converged=converged, params=cur.copy(), start=np.array(case["dev_start"]))


In [4]:
results = []
for case in CASES:
    print("=" * 70)
    results.append(run_case(case, verbose=True))


[idx0] SNR=44.61  overlap(start, NM ov=0.998290) = 0.999994
Body is not plunging, Fisher should be stable.
waveform shape: (3, 6311629)
Computing SNR for parameters: (np.float64(1000148.67), np.float64(10.0010513), np.float64(0.90009637), np.float64(7.49942614), np.float64(0.499974688), 1.0, 5.0, np.float64(0.78338829), np.float64(0.999626083), 1.0, 1.0471975511965976, np.float64(0.909269038), 0.5, np.float64(0.396038198), 0.0, False, False, False, True, 0.0, 0.0, np.float64(0.0), np.float64(0.0))
Waveform Generated. SNR: 44.66124272368732
calculating stable deltas...
minimum relative error is greater than 1% for dev_1. Fisher may be unstable!
minimum relative error is greater than 1% for dev_2. Fisher may be unstable!
Time taken to compute stable deltas is 57.10844278335571 seconds
calculating Fisher matrix...
Finished derivatives
Calculated Fisher is *atleast* positive-definite.
Time taken to compute FM is 2.960958480834961 seconds
[idx0]  it   0 lam=6.7e-03 ov=0.9999939 chi2=2.466e-

In [5]:
print(f"{'case':6} {'SNR':>7} {'ov@start':>11} {'ov@CV':>11} {'ov@NM':>11} {'improved?':>10}")
for r in results:
    imp = "YES" if r["ov_final"] > r["ov_start"] + 1e-8 else "no"
    print(f"{r['name']:6} {r['snr']:>7.2f} {r['ov_start']:>11.7f} {r['ov_final']:>11.7f} "
          f"{r['dev_nm_overlap']:>11.7f} {imp:>10}")

print("\nCV shift of the deviation params (dev_1=del_0_p, dev_2=del_0_e):")
for r in results:
    d1s, d2s, d1f, d2f = r["start"][9], r["start"][10], r["params"][9], r["params"][10]
    print(f"[{r['name']}] dev_1: {d1s:+.4e} -> {d1f:+.4e}   dev_2: {d2s:+.4e} -> {d2f:+.4e}")

print("\noptimized 11-param points  [m1, m2, a, p0, e0, qS, phiS, Phi_phi0, Phi_r0, dev_1, dev_2]:")
for r in results:
    vals = ", ".join(f"{v:.8e}" for v in r["params"])
    print(f"[{r['name']}] = [{vals}]")


case       SNR    ov@start       ov@CV       ov@NM  improved?
idx0     44.61   0.9999939   0.9999954   0.9982902        YES
idx9     55.01   0.9981785   0.9981800   0.9980454        YES
idx13    39.15   0.9997168   0.9997569   0.9998873        YES

CV shift of the deviation params (dev_1=del_0_p, dev_2=del_0_e):
[idx0] dev_1: +0.0000e+00 -> +7.0868e-05   dev_2: +0.0000e+00 -> -4.4712e-05
[idx9] dev_1: +0.0000e+00 -> -4.6866e-05   dev_2: +0.0000e+00 -> -7.2053e-06
[idx13] dev_1: +0.0000e+00 -> -3.5870e-04   dev_2: +0.0000e+00 -> -1.6617e-03

optimized 11-param points  [m1, m2, a, p0, e0, qS, phiS, Phi_phi0, Phi_r0, dev_1, dev_2]:
[idx0] = [1.00014867e+06, 1.00010513e+01, 9.00096370e-01, 7.49942614e+00, 4.99974689e-01, 7.83372816e-01, 9.99625682e-01, 9.09301395e-01, 3.96111386e-01, 7.08678882e-05, -4.47115810e-05]
[idx9] = [9.99994929e+05, 1.00000755e+01, 9.00001698e-01, 9.07417360e+00, 1.99997409e-01, 1.04655069e+00, 7.83181937e-01, 1.59479943e-01, 9.90395355e-02, -4.68661006e-05, -7.20

case       SNR    ov@start       ov@CV       ov@NM  improved?
idx0     44.61   0.9999939   0.9999954   0.9982902        YES
idx9     55.01   0.9981785   0.9981800   0.9980454        YES
idx13    39.15   0.9997168   0.9997569   0.9998873        YES

CV shift of the deviation params (dev_1=del_0_p, dev_2=del_0_e):
[idx0] dev_1: +0.0000e+00 -> +7.0868e-05   dev_2: +0.0000e+00 -> -4.4712e-05
[idx9] dev_1: +0.0000e+00 -> -4.6866e-05   dev_2: +0.0000e+00 -> -7.2053e-06
[idx13] dev_1: +0.0000e+00 -> -3.5870e-04   dev_2: +0.0000e+00 -> -1.6617e-03

optimized 11-param points  [m1, m2, a, p0, e0, qS, phiS, Phi_phi0, Phi_r0, dev_1, dev_2]:
[idx0] = [1.00014867e+06, 1.00010513e+01, 9.00096370e-01, 7.49942614e+00, 4.99974689e-01, 7.83372816e-01, 9.99625682e-01, 9.09301395e-01, 3.96111386e-01, 7.08678882e-05, -4.47115810e-05]
[idx9] = [9.99994929e+05, 1.00000755e+01, 9.00001698e-01, 9.07417360e+00, 1.99997409e-01, 1.04655069e+00, 7.83181937e-01, 1.59479943e-01, 9.90395355e-02, -4.68661006e-05, -7.20533123e-06]
[idx13] = [1.00003623e+06, 1.00001339e+01, 5.00052132e-01, 9.97041651e+00, 2.99991840e-01, 1.04532674e+00, 7.83890513e-01, 1.13528623e-01, 2.53382862e-01, -3.58696791e-04, -1.66173871e-03]